In [1]:
import json
import jsonlines
import logging
import os
import pandas as pd
import time

from musicxmatch_api import MusixMatchAPI

In [2]:
package_root_dir = os.path.join(os.getcwd(), "..")

In [3]:
FantAIno_df = pd.read_csv(os.path.join(package_root_dir, "data", "processed", "melondy_w_dummy_genres.csv"))
FantAIno_df.head()

,artist,album,image_url,rating,is_garage rock,is_hardcore punk,is_punk,is_rock,is_pop,is_art rock,...,is_electro,is_r&b,is_southern hip hop,is_dream pop,is_shoegaze,is_gangsta rap,is_alternative dance,is_punk rock,is_pop punk,is_emo
0,おとぼけビ〜バ〜,SUPER CHAMPON,https://d1j3ls2jacen4o.cloudfront.net/thumbnai...,8,True,True,True,True,False,False,...,False,False,False,False,False,False,False,False,False,False
1,Carly Rae Jepsen,The Loveliest Time,https://d1j3ls2jacen4o.cloudfront.net/thumbnai...,6,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
2,Kayo Dot,Coffins on Io,https://d1j3ls2jacen4o.cloudfront.net/thumbnai...,6,False,False,False,True,False,True,...,False,False,False,False,False,False,False,False,False,False
3,PJ Harvey,Let England Shake,https://d1j3ls2jacen4o.cloudfront.net/thumbnai...,8,False,False,False,True,True,True,...,False,False,False,False,False,False,False,False,False,False
4,Royce da 5′9″,The Allegory,https://d1j3ls2jacen4o.cloudfront.net/thumbnai...,6,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [4]:
FantAIno_df.shape

(3293, 78)

## MusicXMatch API Demo

In [5]:
api = MusixMatchAPI()
search = api.search_artist("Eminem")
print(json.dumps(search, indent=4))
with open('MusicXMatch_API_Demo.json', 'w') as f:
    json.dump(search, f, indent=4)
print(len(search["message"]["body"]["artist_list"]))

JSONDecodeError: Expecting value: line 2 column 1 (char 1)

In [ ]:
search = api.get_artist_albums(65395545)
print(json.dumps(search, indent=4))

In [ ]:
search = api.get_album_tracks(87190259)
print(json.dumps(search, indent=4))

In [ ]:
search = api.get_track(360520429)
print(json.dumps(search, indent=4))

### What can you do with this API?

In [ ]:
print(search)

In [ ]:
dir(api)

Here is an attempt at extracting all of the tracks given an album. This information will prove useful to extract lyrics.

In [8]:
logger = logging.getLogger(__name__)
logging.basicConfig(filename='extract_lyrics.log', encoding='utf-8', level=logging.DEBUG)

if not os.path.isfile('lyrics.jsonl'):
    with open('lyrics.jsonl', 'w', encoding='utf-8') as f:
        json.dump([], f)

cached_lyrics = set()

for index, FantAIno_entry in FantAIno_df.iterrows():
    
    # don't use API credits on processed albums
    with open('lyrics.jsonl', mode='r', encoding='utf-8') as cached_lyrics_reader:
        for line in cached_lyrics_reader:
            if json.loads(line) == []:
                break
            else:
                cached_lyrics.add((json.loads(line)['artist'], json.loads(line)['album']))
    if (FantAIno_entry['artist'], FantAIno_entry['album']) in cached_lyrics:
        logger.info(f"Skipping {FantAIno_entry['artist']} - {FantAIno_entry['album']} is already cached.")
        continue

    # extract artist ID
    try:
        artist_search = api.search_artist(FantAIno_entry["artist"])
        if artist_search["message"]["header"]["status_code"] != 200:
            artist_search = api.search_artist(FantAIno_entry["artist"].lower())
        artist_search_results = artist_search['message']['body']['artist_list']
        for artist_obj in artist_search_results:
            # lots of artists use the same name, so artist-vanity-id is a verification check
            artist = artist_obj['artist']
            artist_vanity_name = artist['artist_vanity_id'].replace("-", " ")
            if artist['artist_name'].lower() == FantAIno_entry["artist"].lower() and artist['artist_name'].lower() == artist_vanity_name.lower():
                artist_id = artist['artist_id']
                break
        else:
            logger.info(f"Artist '{FantAIno_entry['artist']}' not found")
            time.sleep(5)
            continue
    except Exception as e:
        logger.error(f"Error finding artist ID for {FantAIno_entry['artist']}: {repr(e)}. The output of the API call is: {artist_search}")
        time.sleep(5)
        continue    

    # get the artist's albums
    try:
        album_search = api.get_artist_albums(artist_id)
        if album_search["message"]["header"]["status_code"] != 200:
            album_search = api.get_artist_albums(artist_id.lower())
        album_search_results = album_search['message']['body']['album_list']
        for album_obj in album_search_results:
            album = album_obj['album']
            if album['album_name'].lower() == FantAIno_entry["album"].lower():
                album_id = album['album_id']
                break
        else:
            logging.info(f"Album '{FantAIno_entry['album']}' not found")
            time.sleep(5)
            continue
    except Exception as e:
        logger.error(f"Error finding artist ID for {FantAIno_entry['album']}: {repr(e)}. The output of the API call is: {album_search}")
        time.sleep(5)
        continue

    # extract the tracks from album
    try:
        tracks_search = api.get_album_tracks(album_id)
        track_search_results = tracks_search['message']['body']['track_list']
    except Exception as e:
        logger.error(f"Error finding artist tracks for {FantAIno_entry['album']}: {repr(e)}. The output of the API call is: {tracks_search}")
        time.sleep(5)
        continue
    
    
    # get the lyrics
    try:
        for track_obj in track_search_results:
            track = track_obj['track']
            lyrics_search = api.get_track_lyrics(track['track_id'])
            lyrics = lyrics_search['message']['body']['lyrics']['lyrics_body']
            track_lyrics_obj = {
                "artist": FantAIno_entry["artist"],
                "album": FantAIno_entry["album"],
                "tracks": {}
            }
            track_lyrics_obj["tracks"][track['track_name']] = lyrics
            with jsonlines.open('lyrics.jsonl', mode='a') as writer:
                writer.write(track_lyrics_obj)
    except Exception as e:
        logger.error(f"Error getting lyrics for {FantAIno_entry['album']}: {repr(e)}. The output of the API call is: {lyrics_search}")
        time.sleep(5)
        continue

    print(f"Finished processing artist & album: {FantAIno_entry['artist']}, {FantAIno_entry['album']}")

NameError: name 'artist_search' is not defined

In [ ]:
FantAIno_df["tracks_hash"].isna().sum()

In [ ]:
artist_obj

# Get lyrics by Spotify ID.